# Synthetic Dataset Generator for Solar-Powered IoT Nodes

This notebook generates a synthetic dataset to evaluate design configurations of solar-powered IoT nodes.  
Simplified model includes: 

- Real hourly irradiance and temperature data (8760 hours).
- Photovoltaic panels of multiple realistic sizes (1–400 cm²).
- Battery capacities expressed in mAh, typical for IoT devices.
- Temperature losses in the PV module, PMU conversion losses, battery charge/discharge efficiency, and minimum allowed SOC.
- Hour-by-hour simulation of the battery State of Charge (SoC) for an entire year.
- Aggregated viability metrics per configuration.

The goal is to identify panel–battery combinations that provide sufficient energy autonomy under realistic environmental conditions.


## Workflow

### **C1 — setup_constants**
Defines:
- Node power consumption.
- PV panel performance parameters (η_STC, γ, NOCT).
- PMU efficiency.
- Battery constants (nominal voltage, charge/discharge efficiency, SOC minimum).
- Realistic list of PV panel areas (1–400 cm²).
- Realistic list of battery capacities in mAh.

### **C2 — build_design_space**
Creates all possible combinations:

(panel_area_m2 × battery_capacity_mAh × pmu_eta)

Each combination represents one independent system configuration.

### **C3 — load_irradiance_data**
Loads the file `irradiance.csv` containing:

Month, Day, Hour, Gh_W_m2, Tamb_C

The simulation does not use actual dates or years; it assumes a continuous sequence of hours.

### **C4 — compute_pv_power**
For each panel size and hour, computes:
- PV module temperature using the NOCT model.
- Temperature-corrected PV efficiency.
- PV electrical power output (`Ppv_W`).

Produces a long-format dataset (`df_pv`) with one row per:

hour × panel_area_m2

### **C5 — compute_hourly_balance**
Computes:
- PMU-adjusted power (`Ppmu_W`).
- Net power balance:

P_net_W = Ppmu_W − NODE_POWER_W

- Net charging/discharging current using the battery voltage:

I_net_mA = (P_net_W / BATTERY_VOLTAGE) × 1000

### **C6 — simulate_battery_soc**
Simulates the battery SoC hour by hour for every configuration:
- Iterates over all (panel × battery) combinations.
- Integrates the net current in mAh.
- Applies charge/discharge efficiency.
- Enforces SOC bounds: `SOC_MIN ≤ SoC ≤ 1`.

The resulting dataset `df_soc` contains:

hour_index, panel_area_m2, battery_capacity_mAh, SoC, I_net_mA, ...

### **C7 — evaluate_viability**
Computes aggregated performance metrics per configuration, including:
- Fraction of time at SOC minimum (`soc_min_fraction`)
- Fraction of time at full SOC (`soc_full_fraction`)
- Surplus and deficit current-hours (`surplus_mAh`, `deficit_mAh`)
- Net energy balance (`net_mAh`)
- **Longest continuous autonomy** before reaching SOC_MIN (`autonomy_hours`)
- Mean and standard deviation of SoC over the year (`soc_mean`, `soc_std`)

The result is the table `summary`, used for interpretation and visual analysis.

## Input Files

- **irradiance.csv**  
  Hourly irradiance (Gh) and temperature (Tamb) data used for simulation.

## Final Notes

This pipeline is designed to:
- Explore a wide configuration space efficiently.
- Support future extensions (panel tilt, variable loads, PMU models, etc.).
- Remain scalable thanks to long-format data representation.



In [19]:
import numpy as np
import pandas as pd
import itertools
import seaborn as sns
import matplotlib.pyplot as plt

## Column definitions 

### Dataset `df_pv`  
*(One row per hour × panel_area_m2)*

| Column name       | Unit | Description |
|-------------------|------|-------------|
| `hour_index`      | —    | Sequential hour index starting at 0 (0–8759) |
| `Month`           | —    | Calendar month from input dataset |
| `Day`             | —    | Calendar day from input dataset |
| `Hour`            | —    | Hour of day (0–23) from input dataset |
| `panel_area_m2`   | m²   | Effective PV panel area for this record |
| `Gh_W_m2`         | W/m² | Global horizontal irradiance |
| `Tamb_C`          | °C   | Ambient temperature |
| `Tmod_C`          | °C   | PV module temperature (NOCT model) |
| `eta_pv`          | —    | Temperature-corrected PV efficiency |
| `Ppv_W`           | W    | PV power before PMU |
  

### Dataset `df_pv_pmu`  
*(df_pv expanded for every PMU efficiency)*

| Column name       | Unit | Description |
|-------------------|------|-------------|
| `pmu_eta`         | —    | PMU efficiency value applied to this row |
| `Ppmu_W`          | W    | PV power after PMU losses (`Ppv_W × pmu_eta`) |
| `P_net_W`         | W    | Net power delivered to the battery: `Ppmu_W − NODE_POWER_W` |
| `I_net_mA`        | mA   | Net battery charge/discharge current |

*(All previous df_pv columns are inherited.)*


### Dataset `df_soc`  
*(One row per hour × panel × battery × pmu_eta)*

| Column name            | Unit | Description |
|------------------------|------|-------------|
| `hour_index`           | —    | Sequential hour index |
| `Month`                | —    | Calendar month |
| `Day`                  | —    | Calendar day |
| `Hour`                 | —    | Hour of day |
| `panel_area_m2`        | m²   | PV panel area |
| `Gh_W_m2`              | W/m² | Global horizontal irradiance |
| `Tamb_C`               | °C   | Ambient temperature |
| `Tmod_C`               | °C   | PV module temperature |
| `eta_pv`               | —    | PV efficiency (temperature-corrected) |
| `Ppv_W`                | W    | PV power before PMU |
| `pmu_eta`              | —    | PMU efficiency for this record |
| `Ppmu_W`               | W    | PMU output power |
| `P_net_W`              | W    | Net power after subtracting node consumption |
| `I_net_mA`             | mA   | Net current into battery (charging/discharging) |
| `battery_capacity_mAh` | mAh  | Nominal battery capacity for this record |
| `SoC`                  | —    | State of Charge (0–1) |


### Dataset `summary`  
*(One row per configuration: panel × battery × pmu_eta)*

| Column name            | Unit | Description |
|------------------------|------|-------------|
| `panel_area_m2`        | m²   | PV panel area |
| `battery_capacity_mAh` | mAh  | Battery capacity |
| `pmu_eta`              | —    | PMU efficiency |
| `hours_total`          | h    | Total simulated hours (typically 8760) |
| `hours_soc_min`        | h    | Hours at or below SOC_MIN threshold |
| `hours_soc_full`       | h    | Hours at full charge (SoC = 1.0) |
| `soc_mean`             | —    | Mean state of charge |
| `soc_std`              | —    | Standard deviation of SoC |
| `surplus_mAh`          | mAh  | Total charge-hours where I_net_mA > 0 |
| `deficit_mAh`          | mAh  | Total discharge-hours where I_net_mA < 0 |
| `net_mAh`              | mAh  | `surplus_mAh − deficit_mAh` |
| `autonomy_hours`       | h    | Longest continuous run above SOC_MIN |
| `soc_min_fraction`     | —    | Fraction of hours at SOC_MIN |
| `soc_full_fraction`    | —    | Fraction of hours at full charge |


## setup_constants

In [20]:
# --- Node parameters ---
NODE_POWER_W = 0.05  # Constant node power consumption (50 mW)

# --- Photovoltaic panel constants ---
ETA_STC = 0.175        # Efficiency at STC
GAMMA_PER_C = -0.0045  # Temperature coefficient (%/°C)
NOCT_C = 45.0          # Nominal Operating Cell Temperature (°C)

# --- Battery constants ---
BATTERY_ETA_C = 0.95  # Charge/discharge efficiency
SOC_MIN = 0.2         # Minimum allowed SoC (fraction)
BATTERY_VOLTAGE = 3.7  # Nominal voltage (V)

# --- Variable parameter ranges ---

pmu_eta_values = [0.87, 0.90, 0.95, 0.98]

# Photovoltaic panel areas (1–400 cm²)
panel_areas_m2 = [
    0.0001,   # 1 cm²
    0.00025,  # 2.5 cm²
    0.0004,   # 4 cm²
    0.000625, # 6.25 cm² (≈2.5×2.5 cm)
    0.0010,   # 10 cm²
    0.0025,   # 25 cm²
    0.0040,   # 40 cm²
    0.00625,  # 62.5 cm² (≈8×8 cm cell)
    0.0080,   # 80 cm²
    0.0100,   # 100 cm² (≈10×10 cm)
    0.0160,   # 160 cm² (≈12.6×12.6 cm)
    0.0250,   # 250 cm² (≈15.8×15.8 cm)
    0.0310,   # 310 cm² (≈17.6×17.6 cm)
    0.0400    # 400 cm² (≈20×20 cm)
]

# Battery capacities (realistic IoT battery sizes)
battery_capacities_mAh = [
    30,    # small Li-ion coin cell (~0.1 Wh @ 3.7 V)
    70,    # supercap or very small LiPo (~0.25 Wh @ 3.7 V)
    135,   # compact LiPo (~0.5 Wh @ 3.7 V)
    270,   # small pouch cell (~1.0 Wh @ 3.7 V)
    500,   # standard Li-ion (~2.0 Wh @ 3.7 V)
    1000,  # one 18650 cell (~3.7 Wh)
    1300,  # small LiPo pack (~5.0 Wh @ 3.7 V)
    2000,  # two small Li-ion cells (~7.4 Wh @ 3.7 V)
    2600,  # high-capacity 18650 (~10 Wh)
    4000,  # larger LiPo pack (~15 Wh)
    5400   # large IoT/multi-day autonomy pack (~20 Wh)
]


## build_design_space

In [21]:
# --- Generate design space: combinations of variable parameters ---
design_space = list(itertools.product(
    panel_areas_m2,
    battery_capacities_mAh,
    pmu_eta_values
))

df_design = pd.DataFrame(design_space, columns=[
    "panel_area_m2",
    "battery_capacity_mAh",
    "pmu_eta"
])

df_design

,panel_area_m2,battery_capacity_mAh,pmu_eta
0,0.0001,30,0.87
1,0.0001,30,0.90
2,0.0001,30,0.95
3,0.0001,30,0.98
4,0.0001,70,0.87
...,...,...,...
611,0.0400,4000,0.98
612,0.0400,5400,0.87
613,0.0400,5400,0.90
614,0.0400,5400,0.95


## load_irradiance_data

In [22]:
# --- Load irradiance and temperature data ---
irr_data = pd.read_csv("irradiance.csv")

# --- Basic sanity checks ---
expected_cols = ["Date-hour", "Month", "Day", "Hour", "G(h)", "Temperature"]
missing_cols = [c for c in expected_cols if c not in irr_data.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

# --- Rename columns for internal consistency ---
irr_data = irr_data.rename(columns={
    "Date-hour": "datetime_str",
    "G(h)": "Gh_W_m2",
    "Temperature": "Tamb_C"
})

# --- Keep only relevant numeric columns ---
irr_data = irr_data[["Month", "Day", "Hour", "Gh_W_m2", "Tamb_C"]].reset_index(drop=True)


irr_data

,Month,Day,Hour,Gh_W_m2,Tamb_C
0,1,1,5,0,2.81
1,1,1,6,0,2.28
2,1,1,7,0,1.74
3,1,1,8,61,2.17
4,1,1,9,222,2.45
...,...,...,...,...,...
5470,12,31,15,222,10.36
5471,12,31,16,75,9.73
5472,12,31,17,0,9.19
5473,12,31,18,0,8.66


## compute_pv_power

In [23]:
# --- Build a long-format DataFrame: one row per hour and panel area ---

# Duplicate irradiance data for each panel area
df_pv = pd.concat(
    [irr_data.assign(panel_area_m2=A) for A in panel_areas_m2],
    ignore_index=True
)

# Compute module temperature based on NOCT model
df_pv["Tmod_C"] = df_pv["Tamb_C"] + (NOCT_C - 20.0) / 800.0 * df_pv["Gh_W_m2"]

# Compute temperature-corrected panel efficiency
df_pv["eta_pv"] = ETA_STC * (1.0 + GAMMA_PER_C * (df_pv["Tmod_C"] - 25.0))

# Compute PV power output
df_pv["Ppv_W"] = df_pv["Gh_W_m2"] * df_pv["panel_area_m2"] * df_pv["eta_pv"]

# Add hour index (0..n-1)
df_pv["hour_index"] = df_pv.groupby("panel_area_m2").cumcount()

# Keep relevant columns
df_pv = df_pv[[
    "hour_index",
    "Month",
    "Day",
    "Hour",
    "panel_area_m2",
    "Gh_W_m2",
    "Tamb_C",
    "Tmod_C",
    "eta_pv",
    "Ppv_W"
]]

df_pv

,hour_index,Month,Day,Hour,panel_area_m2,Gh_W_m2,Tamb_C,Tmod_C,eta_pv,Ppv_W
0,0,1,1,5,0.0001,0,2.81,2.81000,0.192475,0.000000
1,1,1,1,6,0.0001,0,2.28,2.28000,0.192892,0.000000
2,2,1,1,7,0.0001,0,1.74,1.74000,0.193317,0.000000
3,3,1,1,8,0.0001,61,2.17,4.07625,0.191477,0.001168
4,4,1,1,9,0.0001,222,2.45,9.38750,0.187295,0.004158
...,...,...,...,...,...,...,...,...,...,...
76645,5470,12,31,15,0.0400,222,10.36,17.29750,0.181066,1.607864
76646,5471,12,31,16,0.0400,75,9.73,12.07375,0.185179,0.555538
76647,5472,12,31,17,0.0400,0,9.19,9.19000,0.187450,0.000000
76648,5473,12,31,18,0.0400,0,8.66,8.66000,0.187868,0.000000


## compute_hourly_balance

In [24]:
# Expand df_pv for all PMU efficiencies
df_pv_pmu = pd.concat(
    [df_pv.assign(pmu_eta=eta) for eta in pmu_eta_values],
    ignore_index=True
)

# Compute PMU-adjusted power
df_pv_pmu["Ppmu_W"] = df_pv_pmu["Ppv_W"] * df_pv_pmu["pmu_eta"]

# Net power (W)
df_pv_pmu["P_net_W"] = df_pv_pmu["Ppmu_W"] - NODE_POWER_W

# Convert net power to net current (mA)
df_pv_pmu["I_net_mA"] = (df_pv_pmu["P_net_W"] / BATTERY_VOLTAGE) * 1000

df_pv_pmu


,hour_index,Month,Day,Hour,panel_area_m2,Gh_W_m2,Tamb_C,Tmod_C,eta_pv,Ppv_W,pmu_eta,Ppmu_W,P_net_W,I_net_mA
0,0,1,1,5,0.0001,0,2.81,2.81000,0.192475,0.000000,0.87,0.000000,-0.050000,-13.513514
1,1,1,1,6,0.0001,0,2.28,2.28000,0.192892,0.000000,0.87,0.000000,-0.050000,-13.513514
2,2,1,1,7,0.0001,0,1.74,1.74000,0.193317,0.000000,0.87,0.000000,-0.050000,-13.513514
3,3,1,1,8,0.0001,61,2.17,4.07625,0.191477,0.001168,0.87,0.001016,-0.048984,-13.238873
4,4,1,1,9,0.0001,222,2.45,9.38750,0.187295,0.004158,0.87,0.003617,-0.046383,-12.535834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306595,5470,12,31,15,0.0400,222,10.36,17.29750,0.181066,1.607864,0.98,1.575706,1.525706,412.353057
306596,5471,12,31,16,0.0400,75,9.73,12.07375,0.185179,0.555538,0.98,0.544428,0.494428,133.629054
306597,5472,12,31,17,0.0400,0,9.19,9.19000,0.187450,0.000000,0.98,0.000000,-0.050000,-13.513514
306598,5473,12,31,18,0.0400,0,8.66,8.66000,0.187868,0.000000,0.98,0.000000,-0.050000,-13.513514


## simulate_battery_soc

In [25]:
# --- Build battery simulation dataset (one row per hour, panel, and battery) ---

# Expand over battery capacities as well
df_soc = pd.concat(
    [df_pv_pmu.assign(battery_capacity_mAh=Cbat) for Cbat in battery_capacities_mAh],
    ignore_index=True
)

df_soc["SoC"] = np.nan

# Simulation per (panel_area, battery_capacity, pmu_eta)
for (A, Cbat, eta), group_idx in df_soc.groupby(
    ["panel_area_m2", "battery_capacity_mAh", "pmu_eta"]
).groups.items():

    idx = list(group_idx)
    i_net = df_soc.loc[idx, "I_net_mA"].to_numpy()

    soc = np.empty_like(i_net)
    soc[0] = 1.0

    for i in range(1, len(i_net)):
        delta = i_net[i]
        if delta >= 0:
            soc[i] = soc[i-1] + (delta / Cbat) * BATTERY_ETA_C
        else:
            soc[i] = soc[i-1] + (delta / Cbat) / BATTERY_ETA_C

        soc[i] = min(1.0, max(SOC_MIN, soc[i]))

    df_soc.loc[idx, "SoC"] = soc

df_soc

,hour_index,Month,Day,Hour,panel_area_m2,Gh_W_m2,Tamb_C,Tmod_C,eta_pv,Ppv_W,pmu_eta,Ppmu_W,P_net_W,I_net_mA,battery_capacity_mAh,SoC
0,0,1,1,5,0.0001,0,2.81,2.81000,0.192475,0.000000,0.87,0.000000,-0.050000,-13.513514,30,1.000000
1,1,1,1,6,0.0001,0,2.28,2.28000,0.192892,0.000000,0.87,0.000000,-0.050000,-13.513514,30,0.525842
2,2,1,1,7,0.0001,0,1.74,1.74000,0.193317,0.000000,0.87,0.000000,-0.050000,-13.513514,30,0.200000
3,3,1,1,8,0.0001,61,2.17,4.07625,0.191477,0.001168,0.87,0.001016,-0.048984,-13.238873,30,0.200000
4,4,1,1,9,0.0001,222,2.45,9.38750,0.187295,0.004158,0.87,0.003617,-0.046383,-12.535834,30,0.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3372595,5470,12,31,15,0.0400,222,10.36,17.29750,0.181066,1.607864,0.98,1.575706,1.525706,412.353057,5400,1.000000
3372596,5471,12,31,16,0.0400,75,9.73,12.07375,0.185179,0.555538,0.98,0.544428,0.494428,133.629054,5400,1.000000
3372597,5472,12,31,17,0.0400,0,9.19,9.19000,0.187450,0.000000,0.98,0.000000,-0.050000,-13.513514,5400,0.997366
3372598,5473,12,31,18,0.0400,0,8.66,8.66000,0.187868,0.000000,0.98,0.000000,-0.050000,-13.513514,5400,0.994732


## evaluate_viability

In [26]:
# --- Evaluate annual performance and viability for each configuration ---
# This step aggregates the results of the SoC simulation (df_soc)
# and produces one row per configuration:
# (panel_area_m2, battery_capacity_mAh, pmu_eta)

def longest_autonomy_hours(soc_series, soc_min):
    """
    Return the longest continuous number of hours during which the SoC stays
    strictly above soc_min. This gives an estimate of how long the system
    can withstand poor irradiance conditions before reaching the minimum SoC.
    """
    below = soc_series <= soc_min + 1e-6
    if np.all(below):
        return 0

    max_len = 0
    current = 0
    for v in below:
        if not v:
            current += 1
            max_len = max(max_len, current)
        else:
            current = 0
    return max_len


# --- Aggregation ---
summary = (
    df_soc
    .groupby(["panel_area_m2", "battery_capacity_mAh", "pmu_eta"], as_index=False)
    .agg(
        hours_total=("SoC", "count"),                      # number of hours simulated (usually 8760)
        hours_soc_min=("SoC", lambda s: np.sum(s <= SOC_MIN + 1e-6)),  # hours at or below SOC_MIN
        hours_soc_full=("SoC", lambda s: np.sum(s >= 1.0 - 1e-6)),     # hours at full capacity
        soc_mean=("SoC", "mean"),                          # mean SoC over the year
        soc_std=("SoC", "std"),                            # standard deviation of SoC
        surplus_mAh=("I_net_mA", lambda s: np.sum(np.clip(s, 0, None))),        # total surplus (charging)
        deficit_mAh=("I_net_mA", lambda s: -np.sum(np.clip(s, None, 0))),       # total deficit (discharging)
        autonomy_hours=("SoC", lambda s: longest_autonomy_hours(s.to_numpy(), SOC_MIN))
    )
)

# --- Derived metrics ---
summary["soc_min_fraction"] = summary["hours_soc_min"] / summary["hours_total"]
summary["soc_full_fraction"] = summary["hours_soc_full"] / summary["hours_total"]
summary["net_mAh"] = summary["surplus_mAh"] - summary["deficit_mAh"]


summary


,panel_area_m2,battery_capacity_mAh,pmu_eta,hours_total,hours_soc_min,hours_soc_full,soc_mean,soc_std,surplus_mAh,deficit_mAh,autonomy_hours,soc_min_fraction,soc_full_fraction,net_mAh
0,0.0001,30,0.87,5475,5473,1,0.200206,0.011673,0.000000e+00,67766.361231,2,0.999635,0.000183,-6.776636e+04
1,0.0001,30,0.90,5475,5473,1,0.200206,0.011673,0.000000e+00,67551.874153,2,0.999635,0.000183,-6.755187e+04
2,0.0001,30,0.95,5475,5473,1,0.200206,0.011673,0.000000e+00,67194.395690,2,0.999635,0.000183,-6.719440e+04
3,0.0001,30,0.98,5475,5473,1,0.200206,0.011673,0.000000e+00,66979.908613,2,0.999635,0.000183,-6.697991e+04
4,0.0001,70,0.87,5475,5470,1,0.200364,0.014733,0.000000e+00,67766.361231,5,0.999087,0.000183,-6.776636e+04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
611,0.0400,4000,0.98,5475,0,4202,0.997872,0.004733,2.744095e+06,15450.325566,5475,0.000000,0.767489,2.728645e+06
612,0.0400,5400,0.87,5475,0,4175,0.998403,0.003525,2.429632e+06,15568.185467,5475,0.000000,0.762557,2.414064e+06
613,0.0400,5400,0.90,5475,0,4185,0.998409,0.003520,2.515392e+06,15533.319919,5475,0.000000,0.764384,2.499858e+06
614,0.0400,5400,0.95,5475,0,4195,0.998419,0.003511,2.658330e+06,15480.184561,5475,0.000000,0.766210,2.642850e+06


## plot_viability_maps

In [27]:
metrics = {
    "soc_min_fraction": "Fraction of time at SOC_MIN",
    "net_mAh": "Net energy balance (mAh)",
    "autonomy_hours": "Maximum continuous autonomy (hours)"
}

for metric, label in metrics.items():
    pivot = summary.pivot(
        index="panel_area_m2",
        columns="battery_capacity_mAh",
        values=metric
    )

    plt.figure(figsize=(10, 6))
    sns.heatmap(
        pivot,
        cmap="viridis",
        annot=True,
        fmt=".2f",
        cbar_kws={"label": label}
    )

    plt.title(f"{label} vs Panel Area and Battery Capacity")
    plt.xlabel("Battery capacity (mAh)")
    plt.ylabel("Panel area (m²)")
    plt.tight_layout()
    plt.show()


ValueError: Index contains duplicate entries, cannot reshape